In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_GetFullLoadWorklist
# MAGIC Selects eligible AUTO_MIGRATE tables and publishes the For Each worklist.

# COMMAND ----------

import json
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("connection_id", "")
dbutils.widgets.text("catalog", "da_accelerators")
dbutils.widgets.text("control_schema", "control")
dbutils.widgets.text("max_tables", "0")
dbutils.widgets.text("only_source_table_ids", "")

run_id         = dbutils.widgets.get("run_id").strip()
connection_id  = dbutils.widgets.get("connection_id").strip()
catalog        = dbutils.widgets.get("catalog").strip()
control_schema = dbutils.widgets.get("control_schema").strip()
max_tables     = int(dbutils.widgets.get("max_tables").strip() or "0")
only_ids_raw   = dbutils.widgets.get("only_source_table_ids").strip()

for name, value in (
    ("run_id", run_id),
    ("connection_id", connection_id),
    ("catalog", catalog),
    ("control_schema", control_schema),
):
    if not value:
        raise ValueError(f"{name} is required")

control_table = f"{catalog}.{control_schema}.source_table_control"

base = spark.table(control_table).filter(
    F.col("connection_id") == F.lit(connection_id)
)

eligible = (
    base
    .filter(F.col("is_active") == F.lit(True))
    .filter(F.upper(F.col("table_decision")) == F.lit("AUTO_MIGRATE"))
    .filter(
        F.coalesce(F.col("initial_load_completed"), F.lit(False))
        == F.lit(False)
    )
    .filter(F.col("target_catalog").isNotNull())
    .filter(F.col("target_schema").isNotNull())
    .filter(F.col("target_table").isNotNull())
)

if only_ids_raw:
    wanted = [x.strip() for x in only_ids_raw.split(",") if x.strip()]
    eligible = eligible.filter(F.col("source_table_id").isin(wanted))
    print(f"Restricted to {len(wanted)} explicitly requested table(s)")

eligible = eligible.select(
    "source_table_id",
    "source_schema",
    "source_table",
    "target_catalog",
    "target_schema",
    "target_table",
).orderBy("source_schema", "source_table")

if max_tables > 0:
    eligible = eligible.limit(max_tables)
    print(f"Limited to first {max_tables} table(s)")

rows = eligible.collect()

# COMMAND ----------

if not rows:
    print("No eligible tables. Diagnostic breakdown for this connection:")
    print(f"  registered          : {base.count()}")
    print(f"  is_active           : {base.filter(F.col('is_active') == True).count()}")
    print(f"  AUTO_MIGRATE        : {base.filter(F.upper(F.col('table_decision')) == 'AUTO_MIGRATE').count()}")
    print(f"  not yet loaded      : {base.filter(F.coalesce(F.col('initial_load_completed'), F.lit(False)) == False).count()}")
    raise ValueError(
        f"No eligible AUTO_MIGRATE tables for connection_id={connection_id!r}. "
        "Confirm Job 1 completed and the tables were activated."
    )

worklist = [
    {
        "run_id": run_id,
        "connection_id": connection_id,
        "source_table_id": r["source_table_id"],
        "source_schema": r["source_schema"],
        "source_table": r["source_table"],
    }
    for r in rows
]

payload_bytes = len(json.dumps(worklist).encode("utf-8"))
print(f"Worklist size : {len(worklist)} table(s), {payload_bytes} bytes")

if payload_bytes > 40000:
    raise ValueError(
        f"Worklist payload is {payload_bytes} bytes and risks the task value "
        "limit. Use max_tables to batch this run."
    )

display(eligible)

dbutils.jobs.taskValues.set(key="worklist", value=worklist)
dbutils.jobs.taskValues.set(key="worklist_count", value=len(worklist))

dbutils.notebook.exit(json.dumps({"count": len(worklist), "run_id": run_id}))